In [ ]:
# ── 0. Imports & Config ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

PROCESSED_DIR = Path('Data/Processed')
MODELS_DIR    = Path('models')
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

MODEL_NAMES = [
    'linear_svm',
    'kernel_svm',
    'cart',
    'random_forest',
    'bagging',
    'knn',
    'naive_bayes',
]

In [ ]:
# ── 1. Load Clothes-Size-Prediction (simple features) ────────────────────────
df_simple = pd.read_csv(PROCESSED_DIR / 'clothes_size_clean.csv')
print(df_simple.shape)
df_simple.head()

In [ ]:
# ── 2. Load RentTheRunway (rich features) ─────────────────────────────────────
df_rich = pd.read_csv(PROCESSED_DIR / 'renttherunway_clean.csv')
print(df_rich.shape)
df_rich.head()

In [ ]:
# ── 3. Align simple feature set ───────────────────────────────────────────────
# Clothes-Size-Prediction: weight, age, height → size (XXS–XXXL)
# Encode size to numeric so saved models can score it

SIMPLE_FEATURES = ['weight', 'age', 'height']
SIMPLE_TARGET   = 'size'

X_simple = df_simple[SIMPLE_FEATURES].dropna()
y_simple  = df_simple.loc[X_simple.index, SIMPLE_TARGET]

size_le = LabelEncoder().fit(y_simple)
y_simple_enc = size_le.transform(y_simple)

print(f'Simple dataset: {X_simple.shape[0]:,} rows  |  classes: {size_le.classes_}')

In [ ]:
# ── 4. Load per-model fit probability CSVs (saved from Phase 2) ───────────────
# These were fitted on RTR — we use them to score the simple dataset
# and compare against RTR test-set performance in the same table.

PROBA_COLS = ['proba_Small', 'proba_Fit', 'proba_Large']

def load_model_results(model_name):
    """Load saved fit probabilities and derive accuracy / macro-F1 on RTR test set."""
    path = PROCESSED_DIR / f'{model_name}_save_fit_probabilities.csv'
    df = pd.read_csv(path, index_col=0)
    y_true = df['true_label']
    y_pred = df['pred_label']
    return {
        'model':          model_name,
        'accuracy':       round(accuracy_score(y_true, y_pred), 4),
        'macro_f1':       round(f1_score(y_true, y_pred, average='macro'), 4),
        'f1_Small':       round(f1_score(y_true, y_pred, average=None, labels=['Small'])[0], 4),
        'f1_Fit':         round(f1_score(y_true, y_pred, average=None, labels=['Fit'])[0], 4),
        'f1_Large':       round(f1_score(y_true, y_pred, average=None, labels=['Large'])[0], 4),
    }

rtr_results = pd.DataFrame([load_model_results(m) for m in MODEL_NAMES])
print('RTR (rich feature) results loaded.')
rtr_results

In [ ]:
# ── 5. Score saved models on Clothes-Size-Prediction (simple features) ────────
# Each saved model was trained on RTR features — here we check how well
# the shared numeric features (weight, age, height) transfer to simple-size prediction.

def score_on_simple(model_name, X, y_enc, label_encoder):
    model = joblib.load(MODELS_DIR / f'{model_name}_fit_classifier.joblib')
    
    # predict using only the shared columns the model knows about
    try:
        preds = model.predict(X)
    except Exception as e:
        print(f'  [{model_name}] Predict failed: {e}')
        return None

    # map integer preds back to size strings if needed
    try:
        preds_str = label_encoder.inverse_transform(preds)
        y_str     = label_encoder.inverse_transform(y_enc)
    except Exception:
        preds_str = preds
        y_str     = y_enc

    return {
        'model':    model_name,
        'accuracy': round(accuracy_score(y_str, preds_str), 4),
        'macro_f1': round(f1_score(y_str, preds_str, average='macro', zero_division=0), 4),
    }

simple_results = pd.DataFrame(
    [r for m in MODEL_NAMES
       if (r := score_on_simple(m, X_simple, y_simple_enc, size_le)) is not None]
)
simple_results

In [ ]:
# ── 6. Side-by-side comparison table ─────────────────────────────────────────
comparison = rtr_results[['model', 'accuracy', 'macro_f1']].rename(columns={
    'accuracy': 'RTR Accuracy',
    'macro_f1': 'RTR Macro-F1',
}).merge(
    simple_results.rename(columns={
        'accuracy': 'Simple Accuracy',
        'macro_f1': 'Simple Macro-F1',
    }),
    on='model'
)

comparison['Accuracy Δ'] = (comparison['RTR Accuracy'] - comparison['Simple Accuracy']).round(4)
comparison['Macro-F1 Δ'] = (comparison['RTR Macro-F1'] - comparison['Simple Macro-F1']).round(4)

comparison.set_index('model', inplace=True)
comparison

In [ ]:
# ── 7. Visualisation ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(comparison))
w = 0.35

for ax, metric, title in zip(
    axes,
    [('RTR Accuracy', 'Simple Accuracy'), ('RTR Macro-F1', 'Simple Macro-F1')],
    ['Accuracy: RTR (Rich) vs Simple Features', 'Macro-F1: RTR (Rich) vs Simple Features']
):
    bars1 = ax.bar(x - w/2, comparison[metric[0]], w, label='RTR (rich)', color='#5b7fba', edgecolor='black', alpha=0.85)
    bars2 = ax.bar(x + w/2, comparison[metric[1]], w, label='Simple',     color='#e07b54', edgecolor='black', alpha=0.85)
    ax.bar_label(bars1, fmt='%.3f', padding=2, fontsize=8)
    ax.bar_label(bars2, fmt='%.3f', padding=2, fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(comparison.index, rotation=20, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'comparison_accuracy_f1_rtr_vs_simple.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8. Delta plot — improvement from rich features ────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

colors = ['#5b9e6d' if v >= 0 else '#c0392b' for v in comparison['Macro-F1 Δ']]
bars = ax.bar(comparison.index, comparison['Macro-F1 Δ'], color=colors, edgecolor='black', alpha=0.85)
ax.bar_label(bars, fmt='%+.3f', padding=3)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Macro-F1 Gain from Rich Feature Set (RTR) vs Simple Baseline')
ax.set_ylabel('Δ Macro-F1  (RTR − Simple)')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'comparison_macrof1_delta.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 9. Per-class F1 on RTR (full breakdown) ───────────────────────────────────
f1_breakdown = rtr_results.set_index('model')[['f1_Small', 'f1_Fit', 'f1_Large']]

f1_breakdown.plot(
    kind='bar', figsize=(11, 5),
    color=['#e07b54', '#5b9e6d', '#5b7fba'],
    edgecolor='black', alpha=0.85
)
plt.title('Per-Class F1 Score — RTR Rich Feature Set')
plt.ylabel('F1 Score')
plt.ylim(0, 1.0)
plt.xticks(rotation=20, ha='right')
plt.legend(title='Class')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'comparison_perclass_f1_rtr.png', dpi=150, bbox_inches='tight')
plt.show()

f1_breakdown